
# Streaming FLIM Phasor and Decay

Online fluorescence decay histogramming and phasor computation for FLIM
using :class:`tttrlib.StreamingDecayHistogram` and
:class:`tttrlib.StreamingPhasor`.

These consumers accept photons one at a time, accumulating microtimes
into per-channel histograms and incremental phasor (g, s) coordinates.
Suitable for real-time FLIM displays during acquisition.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import tttrlib

## Generate synthetic TCSPC microtime data
Simulate a single-exponential decay (τ = 3 ns) sampled at 80 MHz repetition
rate with 4096 microtime bins covering one laser period (12.5 ns).



In [ ]:
n_microtime_bins = 4096
frequency_MHz = 80.0
microtime_resolution = 1e-9 / frequency_MHz * 1e9  # ns per bin
n_photons = 50_000
tau_ns = 3.0

rng = np.random.default_rng(42)
dt = microtime_resolution  # ns per bin

# Exponential decay histogram: probability ∝ exp(-t/tau)
bin_times = np.arange(n_microtime_bins) * dt  # ns
decay = np.exp(-bin_times / tau_ns)
decay /= decay.sum()

microtimes = rng.choice(n_microtime_bins, size=n_photons, p=decay)
channels = rng.integers(0, 2, n_photons).astype(np.int32)  # 2 routing channels

print(f"Simulated {n_photons} photons, τ = {tau_ns} ns, {frequency_MHz} MHz")

## Streaming decay histogram



In [ ]:
hist = tttrlib.StreamingDecayHistogram(
    n_microtime_bins=n_microtime_bins,
    n_channels=2
)

# Feed photons one at a time
for i in range(n_photons):
    hist.push_photon(int(microtimes[i]), int(channels[i]))

decay_ch0 = hist.get_histogram(0)
decay_ch1 = hist.get_histogram(1)
print(f"Channel 0: {hist.get_count(0)} photons")
print(f"Channel 1: {hist.get_count(1)} photons")

## Plot the streaming decay histogram



In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
t_ns = np.arange(n_microtime_bins) * dt
ax.plot(t_ns, decay_ch0, label="Channel 0 (parallel)", alpha=0.8)
ax.plot(t_ns, decay_ch1, label="Channel 1 (perpendicular)", alpha=0.8)
ax.set_xlabel("Time (ns)")
ax.set_ylabel("Counts")
ax.set_title("Streaming decay histogram")
ax.legend()
ax.set_xlim(0, 12)
plt.tight_layout()
plt.show()

## Streaming phasor
Compute the FLIM phasor (g, s) incrementally — one photon at a time.



In [ ]:
phasor = tttrlib.StreamingPhasor(
    frequency_MHz=frequency_MHz,
    n_microtime_bins=n_microtime_bins,
    microtime_resolution=dt * 1e-9  # back to seconds
)

for i in range(n_photons):
    phasor.push_photon(int(microtimes[i]))

g, s, n = phasor.get_phasor()
print(f"Phasor: g = {g:.6f}, s = {s:.6f}, n = {n}")

## Compare to the analytical phasor of a single-exponential decay
For a single exponential τ at repetition frequency f:
  g = 1 / (1 + (2πfτ)²)
  s = 2πfτ / (1 + (2πfτ)²)



In [ ]:
omega = 2 * np.pi * frequency_MHz * 1e6 * tau_ns * 1e-9
g_theory = 1.0 / (1.0 + omega**2)
s_theory = omega / (1.0 + omega**2)
print(f"Theory:  g = {g_theory:.6f}, s = {s_theory:.6f}")
print(f"Error:   Δg = {abs(g - g_theory):.2e}, Δs = {abs(s - s_theory):.2e}")

## Universal circle plot



In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))

# Draw the universal circle
theta = np.linspace(0, np.pi / 2, 100)
circle_g = 0.5 + 0.5 * np.cos(theta)
circle_s = 0.5 * np.sin(theta)
ax.plot(circle_g, circle_s, 'k--', alpha=0.3, label="Universal circle")

ax.plot(g_theory, s_theory, 'r+', markersize=15, label=f"Theory (τ={tau_ns} ns)")
ax.plot(g, s, 'bo', label=f"Streaming ({n:.0f} photons)")
ax.set_xlabel("g (cos)")
ax.set_ylabel("s (sin)")
ax.set_title("Streaming FLIM phasor")
ax.set_xlim(0, 1)
ax.set_ylim(0, 0.7)
ax.set_aspect("equal")
ax.legend()
plt.tight_layout()
plt.show()